# Lesson 12: Prompt engineering — the art of talking to AI 💬
**Learning goals**
1. Master the 4 parts of a good prompt: **role, task, context, format**
2. Learn few-shot prompting (giving examples) and step-by-step thinking
3. Use code to compare different prompts side by side

**⏱️ Timing:** bad vs good 10' | four parts 15' | examples 10' | step by step 10' | prompt battle 15'

> ⚙️ **Before you start:** go to **Runtime → Change runtime type → T4 GPU**, then run the cells from top to bottom.

## 🍔 An analogy first
Walk into a café and say "give me a burger" — the cook has to guess.
Say "**a cheeseburger, no pickles, extra sauce, to take away**" — and you get exactly what you want.

AI is the same: it can't read your mind. **The clearer you are, the better the result.**

In [ ]:
# ⏳ The first run downloads the model (about 1–3 minutes). Be patient!
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL = "Qwen/Qwen2.5-1.5B-Instruct" if torch.cuda.is_available() else "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype="auto", device_map="auto")

def ask(prompt, system="You are a friendly assistant. Answer in simple English.",
        max_new_tokens=300, temperature=0.7):
    """Send a prompt to the AI and return its answer."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": prompt}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(llm.device)
    out = llm.generate(**inputs, max_new_tokens=max_new_tokens,
                       do_sample=temperature > 0, temperature=temperature if temperature > 0 else None)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("✅ Model loaded:", MODEL)

## Part 1: Bad prompt vs good prompt

In [ ]:
bad = "Write a story"
good = """You are a children's author.
Write a story about friendship for 12-year-old students.
Requirements:
- The main character is a penguin who is afraid of water
- About 150 words
- End with one sentence that makes the reader think"""

print("😐 Bad prompt:\n", ask(bad), "\n")
print("🤩 Good prompt:\n", ask(good))

## Part 2: The four-part formula 📐
| Part | Ask yourself | Example |
|---|---|---|
| **Role** | Who should the AI be? | "You are a patient physics teacher" |
| **Task** | What exactly should it do? | "Explain why the sky is blue" |
| **Context** | Who is it for? What's the situation? | "For a 13-year-old who hasn't studied light" |
| **Format** | What should the output look like? | "No more than 5 sentences, end with a multiple-choice question" |

Let's compare with code: add one part at a time and see how the answer changes.

In [ ]:
versions = [
    "Explain why the sky is blue",
    "You are a patient physics teacher. Explain why the sky is blue",
    "You are a patient physics teacher. Explain why the sky is blue to a 13-year-old who hasn't studied light",
    "You are a patient physics teacher. Explain why the sky is blue to a 13-year-old who hasn't studied light. "
    "Use an everyday comparison, no more than 5 sentences, and end with a multiple-choice question.",
]
for i, v in enumerate(versions, 1):
    print(f"===== Version {i} =====\n{ask(v)}\n")

### 🧩 Exercise 1
Pick a topic you care about ("what is a black hole", "how to revise for exams"), write 4 versions of the prompt, and put them in `versions`. Which part made the **biggest** improvement?

## Part 3: Few-shot prompting — show the AI examples 📋
Sometimes it's hard to **describe** what you want, but easy to **give examples**.

In [ ]:
few_shot = """Turn sentences into emojis:
Sentence: I went swimming at the beach today → 🌊🏊‍♂️☀️
Sentence: I ate pizza and watched a movie at home → 🏠🍕🎬
Sentence: It started raining and I forgot my umbrella → """
print(ask(few_shot, temperature=0.3, max_new_tokens=20))

### 🧩 Exercise 2
Use few-shot prompting to build a "**Shakespeare translator**": give 2 examples (modern English → Shakespearean English), then ask the AI to translate "I haven't finished my homework yet".

In [ ]:
my_prompt = """
"""
print(ask(my_prompt, temperature=0.3))

## Part 4: Make the AI think step by step 🪜
Maths and logic questions are easy to get wrong. Asking the AI to **write out its steps first** improves accuracy (especially for small models).

In [ ]:
q = "Sam has 3 boxes of pencils with 12 pencils in each. He gives 4 pencils to each of 5 friends, then buys 1 more box. How many pencils does he have now?"
print("Direct answer:", ask(q + " Answer with just one number.", temperature=0, max_new_tokens=10))
print("\nStep by step:", ask(q + " Work it out step by step, then write 'Answer:'.", temperature=0))
print("\n✅ Correct answer:", 3*12 - 5*4 + 12)

## Part 5: System prompts — give the AI a persona 🎭
The `system` prompt is a "behind-the-scenes instruction" given before the conversation starts. The user doesn't see it, but it shapes every answer.

In [ ]:
pirate = "You are a pirate captain. You always talk like a pirate and love using sailing metaphors."
print(ask("How can I get better at maths?", system=pirate))

### 🚀 Challenge: a tutor that doesn't give away answers
Write a `system` prompt that makes the AI a maths tutor that **only gives hints, never the answer**, and encourages students to think. Test it with classmates' questions — does it ever "leak" the answer?

In [ ]:
tutor = """Write your system prompt here"""
print(ask("2x + 6 = 20. What is x?", system=tutor))

## 🏆 Prompt battle (groups)
Starting prompt: **"write something about school"**
Each group takes a turn improving it (only one part per turn), runs it, and the class votes for the best version. Keep every version in the list below as a record.

In [ ]:
rounds = [
    "write something about school",
    # Group 1's improved version:
    # Group 2's improved version:
]
for i, p in enumerate(rounds):
    print(f"===== Version {i}: {p}\n{ask(p)}\n")

## 📌 Summary
**Good prompt = role + task + context + format**, plus two tricks:
- Hard to describe → **give examples** (few-shot)
- Easy to get wrong → **ask it to think step by step**